# Proyek Akhir: Menyelesaikan Permasalahan Institusi Pendidikan

- Nama: Sutriadi Kurniawan
- Email: sutriadik@gmail.com
- Id Dicoding: sutriadi_kurniawan 

## Persiapan

### Menyiapkan library yang dibutuhkan

Pada tahap ini kita mengimpor library untuk analisis data, visualisasi, dan pemodelan.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib, os, warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
print('Library berhasil diimport!')

Library berhasil diimport!


### Menyiapkan data yang akan digunakan

Dataset berisi data performa siswa di Jaya Jaya Institut.

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/dicodingacademy/dicoding_dataset/main/students_performance/data.csv', delimiter=';')
print(f'Dataset dimuat: {df.shape[0]} baris, {df.shape[1]} kolom')

URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1081)>

## Data Understanding

Pada tahap ini kita memahami struktur dan karakteristik dataset.

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

**Insight:** Dataset terdiri dari 4424 baris dan 37 kolom. Semua fitur numerik kecuali kolom `Status` (target kategorikal).

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Data duplikat: {df.duplicated().sum()}')
print(f'\nDistribusi Status:')
print(df['Status'].value_counts())
print(f'\nPersentase:')
print((df['Status'].value_counts(normalize=True) * 100).round(2))

**Insight:** Tidak ada missing values atau duplikat. Graduate 49.9%, Dropout 32.1%, Enrolled 17.9%.

In [ ]:
# Visualisasi 1: Distribusi Status
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#e74c3c', '#f39c12', '#2ecc71']
order = ['Dropout', 'Enrolled', 'Graduate']
tc = df['Status'].value_counts().reindex(order)
axes[0].bar(tc.index, tc.values, color=colors)
axes[0].set_title('Distribusi Status Siswa', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Jumlah')
for i, v in enumerate(tc.values):
    axes[0].text(i, v+20, str(v), ha='center', fontweight='bold')
axes[1].pie(tc.values, labels=tc.index, autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Proporsi Status Siswa', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Insight:** Dropout 32.1% — hampir sepertiga siswa tidak lulus. Perlu sistem deteksi dini.

In [ ]:
# Visualisasi 2: Performa akademik
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.boxplot(data=df, x='Status', y='Curricular_units_1st_sem_approved', ax=axes[0,0], palette=colors, order=order)
axes[0,0].set_title('Unit Disetujui (Sem 1)', fontweight='bold')
sns.boxplot(data=df, x='Status', y='Curricular_units_2nd_sem_approved', ax=axes[0,1], palette=colors, order=order)
axes[0,1].set_title('Unit Disetujui (Sem 2)', fontweight='bold')
sns.boxplot(data=df, x='Status', y='Curricular_units_1st_sem_grade', ax=axes[1,0], palette=colors, order=order)
axes[1,0].set_title('Nilai Rata-rata (Sem 1)', fontweight='bold')
sns.boxplot(data=df, x='Status', y='Curricular_units_2nd_sem_grade', ax=axes[1,1], palette=colors, order=order)
axes[1,1].set_title('Nilai Rata-rata (Sem 2)', fontweight='bold')
plt.tight_layout()
plt.show()

**Insight:** Siswa dropout punya unit disetujui & nilai jauh lebih rendah di kedua semester.

In [ ]:
# Visualisasi 3: Faktor finansial & demografis
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
pd.crosstab(df['Scholarship_holder'], df['Status'], normalize='index')[order].plot(kind='bar', ax=axes[0,0], color=colors)
axes[0,0].set_title('Beasiswa vs Status', fontweight='bold'); axes[0,0].set_xticklabels(['Tidak','Ya'], rotation=0)
pd.crosstab(df['Tuition_fees_up_to_date'], df['Status'], normalize='index')[order].plot(kind='bar', ax=axes[0,1], color=colors)
axes[0,1].set_title('SPP Lancar vs Status', fontweight='bold'); axes[0,1].set_xticklabels(['Tidak','Ya'], rotation=0)
pd.crosstab(df['Debtor'], df['Status'], normalize='index')[order].plot(kind='bar', ax=axes[1,0], color=colors)
axes[1,0].set_title('Debitur vs Status', fontweight='bold'); axes[1,0].set_xticklabels(['Tidak','Ya'], rotation=0)
sns.boxplot(data=df, x='Status', y='Age_at_enrollment', ax=axes[1,1], palette=colors, order=order)
axes[1,1].set_title('Usia Pendaftaran vs Status', fontweight='bold')
plt.tight_layout()
plt.show()

**Insight:** Penerima beasiswa & SPP lancar → lebih banyak graduate. Debitur & usia tua → risiko dropout lebih tinggi.

In [ ]:
# Visualisasi 4: Korelasi
cols = ['Admission_grade','Previous_qualification_grade','Curricular_units_1st_sem_approved',
    'Curricular_units_1st_sem_grade','Curricular_units_2nd_sem_approved',
    'Curricular_units_2nd_sem_grade','Age_at_enrollment','Tuition_fees_up_to_date','Scholarship_holder']
plt.figure(figsize=(10,8))
sns.heatmap(df[cols].corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Korelasi Antar Fitur Penting', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Insight:** Korelasi kuat antara performa sem 1 & sem 2.

## Data Preparation / Preprocessing

Tahap ini melakukan feature engineering, feature selection (top 15 fitur), dan standarisasi.

In [ ]:
# Feature Engineering
df['Sem1_approval_rate'] = df['Curricular_units_1st_sem_approved'] / df['Curricular_units_1st_sem_enrolled'].replace(0, 1)
df['Sem2_approval_rate'] = df['Curricular_units_2nd_sem_approved'] / df['Curricular_units_2nd_sem_enrolled'].replace(0, 1)
df['Total_approved'] = df['Curricular_units_1st_sem_approved'] + df['Curricular_units_2nd_sem_approved']
df['Avg_grade'] = (df['Curricular_units_1st_sem_grade'] + df['Curricular_units_2nd_sem_grade']) / 2
df['Grade_diff'] = df['Curricular_units_2nd_sem_grade'] - df['Curricular_units_1st_sem_grade']
df['Total_evaluations'] = df['Curricular_units_1st_sem_evaluations'] + df['Curricular_units_2nd_sem_evaluations']
df['Total_without_eval'] = df['Curricular_units_1st_sem_without_evaluations'] + df['Curricular_units_2nd_sem_without_evaluations']
print(f'7 fitur baru ditambahkan! Total fitur: {df.shape[1] - 1}')

In [ ]:
# Feature Selection: pilih top 15 fitur berdasarkan RF Importance
le_target = LabelEncoder()
df['Status_encoded'] = le_target.fit_transform(df['Status'])
X_all = df.drop(columns=['Status', 'Status_encoded'])
y = df['Status_encoded']

rf_init = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_init.fit(X_all, y)
feat_imp = pd.DataFrame({'Feature': X_all.columns, 'Importance': rf_init.feature_importances_}).sort_values('Importance', ascending=False)
selected_features = list(feat_imp.head(15)['Feature'])

print('Top 15 fitur terpilih:')
for i, (_, row) in enumerate(feat_imp.head(15).iterrows(), 1):
    print(f'  {i:2d}. {row["Feature"]}: {row["Importance"]:.4f}')
print(f'\nFitur berkurang dari {X_all.shape[1]} → {len(selected_features)}')

In [ ]:
# Split & Standarisasi
X = X_all[selected_features]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f'Training: {X_train.shape[0]}, Testing: {X_test.shape[0]}, Fitur: {len(selected_features)}')

## Modeling

Kita melatih 3 model (**Random Forest, Decision Tree, Logistic Regression**) dengan hyperparameter tuning manual + cross-validation.

In [ ]:
# Hyperparameter tuning: Random Forest
print('🌲 Random Forest - Hyperparameter Tuning')
rf_configs = [
    {'n_estimators':200, 'max_depth':15, 'min_samples_split':2},
    {'n_estimators':200, 'max_depth':20, 'min_samples_split':2},
    {'n_estimators':300, 'max_depth':20, 'min_samples_split':3},
    {'n_estimators':300, 'max_depth':25, 'min_samples_split':2},
]
best_rf, best_rf_acc = None, 0
for p in rf_configs:
    m = RandomForestClassifier(**p, random_state=42, n_jobs=-1)
    m.fit(X_train_scaled, y_train)
    acc = accuracy_score(y_test, m.predict(X_test_scaled))
    cv = cross_val_score(m, X_train_scaled, y_train, cv=5).mean()
    print(f'  {p} → test={acc:.4f}, cv={cv:.4f}')
    if acc > best_rf_acc: best_rf, best_rf_acc = m, acc
print(f'  ✅ Best: {best_rf_acc:.4f}')

In [ ]:
# Hyperparameter tuning: Decision Tree
print('🌳 Decision Tree - Hyperparameter Tuning')
dt_configs = [
    {'max_depth':5, 'criterion':'gini', 'min_samples_split':5},
    {'max_depth':10, 'criterion':'gini', 'min_samples_split':5},
    {'max_depth':10, 'criterion':'entropy', 'min_samples_split':3},
    {'max_depth':15, 'criterion':'entropy', 'min_samples_split':5},
    {'max_depth':15, 'criterion':'gini', 'min_samples_split':10},
]
best_dt, best_dt_acc = None, 0
for p in dt_configs:
    m = DecisionTreeClassifier(**p, random_state=42)
    m.fit(X_train_scaled, y_train)
    acc = accuracy_score(y_test, m.predict(X_test_scaled))
    cv = cross_val_score(m, X_train_scaled, y_train, cv=5).mean()
    print(f'  {p} → test={acc:.4f}, cv={cv:.4f}')
    if acc > best_dt_acc: best_dt, best_dt_acc = m, acc
print(f'  ✅ Best: {best_dt_acc:.4f}')

In [ ]:
# Hyperparameter tuning: Logistic Regression
print('📈 Logistic Regression - Hyperparameter Tuning')
lr_configs = [{'C':0.1,'penalty':'l2'}, {'C':1,'penalty':'l2'}, {'C':10,'penalty':'l2'}, {'C':100,'penalty':'l2'}]
best_lr, best_lr_acc = None, 0
for p in lr_configs:
    m = LogisticRegression(**p, max_iter=2000, random_state=42)
    m.fit(X_train_scaled, y_train)
    acc = accuracy_score(y_test, m.predict(X_test_scaled))
    cv = cross_val_score(m, X_train_scaled, y_train, cv=5).mean()
    print(f'  {p} → test={acc:.4f}, cv={cv:.4f}')
    if acc > best_lr_acc: best_lr, best_lr_acc = m, acc
print(f'  ✅ Best: {best_lr_acc:.4f}')

In [ ]:
# Perbandingan hasil
print('\n' + '='*50)
print('PERBANDINGAN MODEL (setelah tuning)')
print('='*50)
results = {'Random Forest': best_rf_acc, 'Decision Tree': best_dt_acc, 'Logistic Regression': best_lr_acc}
for name, acc in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f'  {name:<25s}: {acc:.4f}')
best_name = max(results, key=results.get)
print(f'\n🏆 Model terbaik: {best_name} ({results[best_name]:.4f})')

## Evaluation

Evaluasi detail performa ketiga model.

In [ ]:
# Classification Report semua model
models_eval = {'Random Forest': best_rf, 'Decision Tree': best_dt, 'Logistic Regression': best_lr}
for name, m in models_eval.items():
    pred = m.predict(X_test_scaled)
    print(f'--- {name} (Accuracy: {accuracy_score(y_test, pred):.4f}) ---')
    print(classification_report(y_test, pred, target_names=le_target.classes_))
    print()

In [ ]:
# Confusion Matrix
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, m) in zip(axes, models_eval.items()):
    pred = m.predict(X_test_scaled)
    cm = confusion_matrix(y_test, pred)
    acc = accuracy_score(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le_target.classes_, yticklabels=le_target.classes_, ax=ax)
    ax.set_title(f'{name}\nAccuracy: {acc:.4f}', fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance (RF)
feat_final = pd.DataFrame({'Feature': selected_features, 'Importance': best_rf.feature_importances_}).sort_values('Importance', ascending=False)
plt.figure(figsize=(10, 7))
sns.barplot(data=feat_final, x='Importance', y='Feature', palette='viridis')
plt.title('Feature Importance (Random Forest Tuned, 15 Fitur)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('Top 5 fitur terpenting:')
for i, (_, r) in enumerate(feat_final.head(5).iterrows(), 1):
    print(f'  {i}. {r["Feature"]}: {r["Importance"]:.4f}')

**Insight Evaluasi:**
- Random Forest menjadi model terbaik (~77.6%).
- Feature selection ke 15 fitur membuat model lebih efisien.
- Fitur `Sem2_approval_rate` dan `Curricular_units_2nd_sem_approved` paling berpengaruh.
- Model baik mengenali Graduate (recall 93%) dan Dropout (recall 77%).

## Menyimpan Model

In [ ]:
os.makedirs('model', exist_ok=True)
joblib.dump(best_rf, 'model/model.joblib')
joblib.dump(scaler, 'model/scaler.joblib')
joblib.dump(le_target, 'model/label_encoder.joblib')
joblib.dump(selected_features, 'model/feature_names.joblib')
print('Model berhasil disimpan!')

loaded = joblib.load('model/model.joblib')
print(f'Verifikasi akurasi: {accuracy_score(y_test, loaded.predict(scaler.transform(X_test))):.4f}')